# 12.12 答案需要同時看圖和聽聲音時怎麼辦？

# 第 12 章：沿用相同介面加入聲音

前置：第10章模態展開與第7章loss mask。頻譜可獨立讀；聲音生成與log-mel都從PyTorch寫起，不依賴Whisper或torchaudio。FSDD爲額外真實語音pilot；本章默認合成音高／時長任務。




In [ ]:
from pathlib import Path
import os
import subprocess
import sys

# Colab 會先下載專案；本機 Jupyter 沿用已開啟的 repo。
try:
    import google.colab
except ImportError:
    in_colab = False
else:
    in_colab = True

if in_colab:
    root = Path("/content/tiny-perceptron-vlm")
    if not root.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/birdhackor/tiny-perceptron-vlm.git", str(root)],
            env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"},
            check=True,
        )
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", str(root)], check=True)
    os.chdir(root)

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("本機請先依 course/first-steps.md 開啟 repo；免安裝練習可使用本節的 Colab 入口")
sys.path.insert(0, str(root))
import torch
from torch import nn
from torch.nn import functional as F

torch.set_num_threads(1)
torch.manual_seed(42)

**先看圖：形狀在圖片，音高在聲音：兩把鑰匙一起用**

替換每個模態，答案只改對應部分；保留新組合。

橘色邊框只提示閱讀順序，靜態標註一直保留。系統的減少動態效果設定會停用動畫。


In [ ]:
from IPython.display import SVG, display

display(SVG(filename=str(root / "course/figures/joint.svg")))

**先想一想：**只提供圖片，能唯一決定「circle,high」嗎？

設計一個答案必須依兩種線索的任務：圖片給形狀、聲音給高低音。各自替換應只改變對應答案部分，新組合留作holdout。

**把直覺寫成數學：**answer=(shape(image),pitch(audio))；單一模態只能獲得其中一項。

**最小實驗：**以下程式只處理這節的問題。


In [ ]:
from tiny_perceptron.model import TinyLM, ModelConfig, masked_loss
from tiny_perceptron.multimodal import MultiModalLM, scene, tone

model = MultiModalLM(TinyLM(ModelConfig(width=8)))
ids = torch.tensor([1, 3, 5, 6, 4, 100, 2])
labels = torch.tensor([-100] * 5 + [100, 2])
out = model(ids, labels, image=scene(), waveform=tone())
loss = masked_loss(out["logits"], out["labels"])
loss.backward()
print(
    out["logits"].shape,
    "圖梯度",
    model.image_projector.weight.grad.norm().item(),
    "音梯度",
    model.audio_projector.weight.grad.norm().item(),
)

**如何讀結果：**這裏只證實兩種入口有梯度。聯合SFT後應報告兩項與聯合exact match，並替換圖／音做消融。

**只改一件事：**只替換聲音，保持圖和問題不變。
